# Fase 4 — Modelagem

Produto de referência: P0131. Treino até 03/10/2019, teste de 04/10 a 31/10/2019 (H = 28).

## Infraestrutura de avaliação

In [1]:
import sys
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
from src.dados import carregar_demanda
from src.metricas import vies, wmape
from src.preparacao import H, remover_descontinuados, separar_treino_teste, serie_produto

SEED = 42
np.random.seed(SEED)
RAIZ = Path.cwd().parent
CAMINHO_FIGURAS = RAIZ / "relatorios" / "figuras"
PRODUTO = "P0131"

painel = remover_descontinuados(carregar_demanda())
serie_p0131 = serie_produto(painel, PRODUTO)
treino, teste = separar_treino_teste(serie_p0131)
real_teste = teste["unidades"]
print(f"Treino: {treino.index.min().date()} a {treino.index.max().date()} ({len(treino)} dias)")
print(f"Teste:  {teste.index.min().date()} a {teste.index.max().date()} ({len(teste)} dias)")

Treino: 2017-01-02 a 2019-10-03 (1005 dias)
Teste:  2019-10-04 a 2019-10-31 (28 dias)


In [2]:
placar = {}
previsoes = {}


def avaliar(nome, previsao):
    previsao = pd.Series(np.clip(np.asarray(previsao, dtype=float), 0, None), index=real_teste.index)
    placar[nome] = {"wmape": wmape(real_teste, previsao), "vies": vies(real_teste, previsao)}
    previsoes[nome] = previsao
    print(f"{nome:<32} | WMAPE: {placar[nome]['wmape']:6.2f}% | Viés: {placar[nome]['vies']:+6.2f}%")

Checagem das funções com valores de resultado conhecido (não entram no placar).

In [3]:
real_ex = np.array([100, 200, 100])
print(f"Previsão perfeita      -> WMAPE {wmape(real_ex, real_ex):.2f}% | viés {vies(real_ex, real_ex):+.2f}%")
print(f"Tudo 10% acima         -> WMAPE {wmape(real_ex, real_ex * 1.1):.2f}% | viés {vies(real_ex, real_ex * 1.1):+.2f}%")
print(f"Tudo 10% abaixo        -> WMAPE {wmape(real_ex, real_ex * 0.9):.2f}% | viés {vies(real_ex, real_ex * 0.9):+.2f}%")
print(f"+50 e -50 se compensam -> WMAPE {wmape(real_ex, [150, 150, 100]):.2f}% | viés {vies(real_ex, [150, 150, 100]):+.2f}%")
print(f"Clip: previsão [-10, 5, 20] vira {np.clip([-10, 5, 20], 0, None).tolist()}")

Previsão perfeita      -> WMAPE 0.00% | viés +0.00%
Tudo 10% acima         -> WMAPE 10.00% | viés +10.00%
Tudo 10% abaixo        -> WMAPE 10.00% | viés -10.00%
+50 e -50 se compensam -> WMAPE 25.00% | viés +0.00%
Clip: previsão [-10, 5, 20] vira [0, 5, 20]


In [4]:
print(f"Dias avaliados no teste:        {len(real_teste)}")
print(f"Demanda real total no teste:    {real_teste.sum():,.0f} unidades")
print(f"Média diária no teste:          {real_teste.mean():.2f} unidades")

Dias avaliados no teste:        28
Demanda real total no teste:    8,158 unidades
Média diária no teste:          291.36 unidades


## Degrau 1 — Naive sazonal (repete a última semana do treino)

In [5]:
ultima_semana = treino["unidades"].iloc[-7:]
previsao_naive = np.tile(ultima_semana.values, H // 7)

alinhamento = pd.DataFrame({
    "dia_teste": real_teste.index[:7].strftime("%d/%m %a"),
    "copiado_de": ultima_semana.index.strftime("%d/%m %a"),
    "valor": ultima_semana.values,
})
print(alinhamento.to_string(index=False))
print()
avaliar("Naive sazonal (7d)", previsao_naive)

dia_teste copiado_de  valor
04/10 Fri  27/09 Fri  296.0
05/10 Sat  28/09 Sat  327.0
06/10 Sun  29/09 Sun  351.0
07/10 Mon  30/09 Mon  273.0
08/10 Tue  01/10 Tue  244.0
09/10 Wed  02/10 Wed  271.0
10/10 Thu  03/10 Thu  240.0

Naive sazonal (7d)               | WMAPE:  10.17% | Viés:  -1.84%


## Degrau 2 — Processo atual: média das últimas 4 semanas

In [6]:
media_4_semanas = treino["unidades"].iloc[-28:].mean()
previsao_media_4s = np.repeat(media_4_semanas, H)
print(f"Média das últimas 4 semanas do treino ({treino.index[-28].date()} a {treino.index[-1].date()}): {media_4_semanas:.2f}")
print()
avaliar("Média 4 semanas (atual)", previsao_media_4s)

Média das últimas 4 semanas do treino (2019-09-06 a 2019-10-03): 302.39

Média 4 semanas (atual)          | WMAPE:  17.18% | Viés:  +3.79%


In [7]:
fim_semana_teste = real_teste.index.dayofweek >= 5
comparacao = []
for nome in ["Naive sazonal (7d)", "Média 4 semanas (atual)"]:
    prev = previsoes[nome]
    for rotulo, mascara in [("dias úteis", ~fim_semana_teste), ("fim de semana", fim_semana_teste)]:
        comparacao.append({"modelo": nome, "dias": rotulo,
                           "real_medio": real_teste[mascara].mean(), "previsto_medio": prev[mascara].mean(),
                           "wmape": wmape(real_teste[mascara], prev[mascara]), "vies": vies(real_teste[mascara], prev[mascara])})
print(pd.DataFrame(comparacao).round(2).to_string(index=False))

print()
print(f"Amplitude prevista (máx − mín) — naive: {previsoes['Naive sazonal (7d)'].max() - previsoes['Naive sazonal (7d)'].min():.0f} | "
      f"média 4s: {previsoes['Média 4 semanas (atual)'].max() - previsoes['Média 4 semanas (atual)'].min():.0f} | "
      f"real: {real_teste.max() - real_teste.min():.0f}")

semanas_janela = treino["unidades"].iloc[-28:].groupby(np.arange(28) // 7).mean()
print("\nMédia de cada uma das 4 semanas da janela usada pelo comprador:")
print(pd.Series(semanas_janela.values, index=[f"semana {i}" for i in range(1, 5)]).round(1).to_string())
print(f"Média da última semana do treino: {treino['unidades'].iloc[-7:].mean():.1f} | média real do teste: {real_teste.mean():.1f}")

                 modelo          dias  real_medio  previsto_medio  wmape   vies
     Naive sazonal (7d)    dias úteis       262.3          264.80  11.06   0.95
     Naive sazonal (7d) fim de semana       364.0          339.00   8.59  -6.87
Média 4 semanas (atual)    dias úteis       262.3          302.39  17.31  15.29
Média 4 semanas (atual) fim de semana       364.0          302.39  16.93 -16.93

Amplitude prevista (máx − mín) — naive: 111 | média 4s: 0 | real: 187

Média de cada uma das 4 semanas da janela usada pelo comprador:
semana 1    327.4
semana 2    306.9
semana 3    289.3
semana 4    286.0
Média da última semana do treino: 286.0 | média real do teste: 291.4


## Degrau 3 — Holt-Winters (tendência e sazonalidade aditivas, período 7)

In [8]:
import warnings
from statsmodels.tsa.holtwinters import ExponentialSmoothing

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    modelo_hw = ExponentialSmoothing(
        treino["unidades"],
        trend="add",
        seasonal="add",
        seasonal_periods=7,
        initialization_method="estimated",
    ).fit()

previsao_hw = modelo_hw.forecast(H)
avaliar("Holt-Winters aditivo", previsao_hw)

print()
print(f"alfa  (nível):        {modelo_hw.params['smoothing_level']:.4f}")
print(f"beta  (tendência):    {modelo_hw.params['smoothing_trend']:.4f}")
print(f"gama  (sazonalidade): {modelo_hw.params['smoothing_seasonal']:.4f}")
print(f"Convergiu: {modelo_hw.mle_retvals.success if hasattr(modelo_hw, 'mle_retvals') else 'n/d'}")

nivel_final = modelo_hw.level.iloc[-1]
tendencia_final = modelo_hw.trend.iloc[-1]
sazonal_final = pd.Series(modelo_hw.season.iloc[-7:].values,
                          index=modelo_hw.season.index[-7:].strftime("%a"))
print(f"\nNo fim do treino: nível {nivel_final:.2f} | tendência {tendencia_final:+.3f} un/dia por dia "
      f"(= {tendencia_final * H:+.1f} un/dia ao longo dos 28 dias)")
print("Índices sazonais finais (un/dia somadas ao nível):")
print(sazonal_final.round(1).to_string())
print(f"\nPrevisão média no teste: {previsao_hw.mean():.1f} | real médio: {real_teste.mean():.1f}")

Holt-Winters aditivo             | WMAPE:   9.30% | Viés:  -3.82%

alfa  (nível):        0.4953
beta  (tendência):    0.0000
gama  (sazonalidade): 0.0260
Convergiu: True

No fim do treino: nível 276.90 | tendência +0.208 un/dia por dia (= +5.8 un/dia ao longo dos 28 dias)
Índices sazonais finais (un/dia somadas ao nível):
data
Fri     2.2
Sat    62.0
Sun    45.9
Mon   -27.8
Tue   -29.4
Wed   -31.6
Thu   -20.1

Previsão média no teste: 280.2 | real médio: 291.4
